In [3]:
# =====================================================================
# STABILIZOWANY SKRYPT TRENINGOWY SLM-AUDIT DLA GOOGLE COLAB (T4 GPU)
# =====================================================================

# 0. Czyszczenie pamięci podręcznej GPU przed startem
import torch
import gc
torch.cuda.empty_cache()
gc.collect()

# 1. Poprawna i czysta instalacja zależności Unsloth i TRL
!pip install -U torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.29" "trl<0.9.0" peft accelerate bitsandbytes

import json
from datasets import load_dataset
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template
from trl import SFTTrainer
from transformers import TrainingArguments
from google.colab import files
import os

# =====================================================================
# 2. GENEROWANIE BAZY WIEDZY EKSPERCKIEJ (CHATML FORMAT)
# =====================================================================
SYSTEM_PROMPT = """Jesteś elitarnym Oficerem ds. Bezpieczeństwa Żywności i Regulacji Prawnych w Quality Audit Enterprise.
Znasz na wylot standardy IFS Food v8, BRCGS Food Safety v9, Codex Alimentarius HACCP, GMP, GHP oraz procedurę PR15.01.
Twoim zadaniem jest natychmiastowa ocena sytuacji na hali, rozstrzyganie zgodności w skali 1-5, egzekwowanie limitów CCP oraz uruchamianie procedur kwarantanny Hold Lot i zatrzymania linii."""

expert_samples = [
    {
        "instruction": "Podczas testu detektora metali CCP1 wzorzec SS 316 2.5 mm nie został wykryty, a ramię odrzutnika nie zadziałało. Co robimy?",
        "output": "🔴 **STATUS: KRYTYCZNE UCHYBIENIE (KO 2 / BRCGS 4.10)**\n\n1. **NATYCHMIASTOWY STOP LINII:** Wstrzymanie pracy linii.\n2. **KWARANTANNA HOLD LOT (HOLD-2026-XXXX):** Blokada 100% wyrobu z ostatnich 2 h od ostatniego poprawnego testu. Oznaczenie palet taśmą żółto-czarną i blokada w WMS.\n3. **SERWIS:** Utrzymanie Ruchu weryfikuje głowicę i rygiel kosza odrzutów.\n4. **RE-TEST:** Po naprawie, test 3 pozycji dla Fe 1.5mm, Non-Fe 2.0mm i SS 2.5mm przed zwolnieniem partii."
    },
    {
        "instruction": "Na Linii Pralin pękła szklana osłona lampy 1.5 metra od otwartej masy czekoladowej. Zastosuj procedurę PR15.01.",
        "output": "🔴 **STATUS: KRYTYCZNE UCHYBIENIE (PR15.01 / CIAŁO OBCE)**\n\n1. **STOP LINII:** Zatrzymanie taśmociągu.\n2. **KWARANTANNA 5 METRÓW:** Wyznaczenie strefy izolacji o promieniu min. 5 m od miejsca stłuczki.\n3. **UTRATA PRODUKTU:** Całkowita utylizacja otwartego surowca w strefie (zakaz przerobu / No Rework).\n4. **SPRZĄTANIE I KONTROLA:** Odkurzacz przemysłowy, inspekcja światłem skośnym przez QA i pisemna zgoda Kierownika Jakości na rozruch."
    },
    {
        "instruction": "Pracownik strefy High Care zgłasza wymioty i biegunkę, ale chce dokończyć zmianę. Jaki jest reżim sanitarny?",
        "output": "🔴 **STATUS: KRYTYCZNE UCHYBIENIE (IFS KO 4 / GHP)**\n\n1. **ODSUNIĘCIE OD PRACY:** Bezzwłoczna ewakuacja pracownika ze strefy produkcyjnej do punktu medycznego.\n2. **KARENCJA 48H:** Obowiązkowa kwarantanna sanitarna min. 48 godzin od ustąpienia objawów bez leków maskujących.\n3. **DEZYNFEKCJA:** Mycie szafek i węzłów sanitarnych.\n4. **REJESTRACJA:** Wpis w Księdze Dyspens PR15.01."
    },
    {
        "instruction": "Wynik wymazu ATP z taśmy po myciu CIP wynosi 45 RLU. Jaka jest decyzja?",
        "output": "🔴 **STATUS: NIESPEŁNIENIE KRYTERIUM (ATP < 30 RLU)**\n\n1. **ZAKAZ ROZRUCHU:** Limit przekroczony (45 RLU > 30 RLU).\n2. **RE-SANITYZACJA:** Powtórzenie mycia chemicznego i dezynfekcji.\n3. **POWTÓRNY WYMAZ:** Start produkcji wyłącznie przy wyniku poniżej 30 RLU."
    }
]

with open("dataset_food_safety.jsonl", "w", encoding="utf-8") as f:
    for s in expert_samples:
        entry = {
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": s["instruction"]},
                {"role": "assistant", "content": s["output"]}
            ]
        }
        f.write(json.dumps(entry, ensure_ascii=False) + "\n")

# =====================================================================
# 3. ŁADOWANIE MODELU (OPTIMIZED FOR T4 GPU)
# =====================================================================
max_seq_length = 1024  # Zmniejszono do 1024 dla oszczędności VRAM na T4
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-7B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

# =====================================================================
# 4. TRENING SFT
# =====================================================================
tokenizer = get_chat_template(tokenizer, chat_template = "chatml")

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize = False, add_generation_prompt = False) for convo in convos]
    return { "text" : texts }

dataset = load_dataset("json", data_files="dataset_food_safety.jsonl", split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 1,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 1,  # Bezpieczne dla T4
        gradient_accumulation_steps = 4,
        warmup_steps = 3,
        max_steps = 40,                   # Szybki test stabilności
        learning_rate = 2e-4,
        fp16 = True,
        bf16 = False,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

print("🚀 Rozpoczynanie stabilizowanego treningu eksperckiego...")
trainer.train()

# =====================================================================
# 5. EKSPORT DO GGUF I POBIERANIE
# =====================================================================
model.save_pretrained_gguf("slm_audit_model", tokenizer, quantization_method = "q4_k_m")

for file in os.listdir("slm_audit_model"):
    if file.endswith(".gguf"):
        gguf_path = os.path.join("slm_audit_model", file)
        print(f"📦 Sukces! Pobieranie pliku wag GGUF: {file}")
        files.download(gguf_path)

Looking in indexes: https://download.pytorch.org/whl/cu118
  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-sfttvg9h/unsloth_892b22697d194b048c2b279bbb10cf4c
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-sfttvg9h/unsloth_892b22697d194b048c2b279bbb10cf4c
  Resolved https://github.com/unslothai/unsloth.git to commit ce99e954cbb690d8a6335ac3d95f43f5881c8493
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.50.2-py3-none-manylinux_2_24_x86_64.whl (43.1 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 34.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Unsloth 2026.8.22 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.
Unsloth: Restored added_tokens_decoder metadata in /content/_unsloth_sentencepiece_temp/tokenizer_lryf6otl/tokenizer_config.json.


Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

...

model-00004-of-00004.safetensors: downloading bytes:           |  0.00B            



Unsloth: Preparing safetensor model files: 100%|██████████| 4/4 [03:20<00:00, 50.21s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)




Unsloth: Merging weights into 16bit:   0%|          | 0/4 [00:00<?, ?it/s]

Unsloth: Merging weights into 16bit:  25%|██▌       | 1/4 [00:50<02:31, 50.47s/it]

Unsloth: Merging weights into 16bit:  50%|█████     | 2/4 [01:48<01:50, 55.02s/it]

Unsloth: Merging weights into 16bit:  75%|███████▌  | 3/4 [02:38<00:52, 52.72s/it]

Unsloth: Merging weights into 16bit: 100%|██████████| 4/4 [02:47<00:00, 41.94s/it]


Unsloth: Merge process complete. Saved to `/content/slm_audit_model`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF f16 might take 3 minutes.
\        /    [2] Converting GGUF f16 to ['q4_k_m'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: Installing llama.cpp. This might take 3 minutes...
Unsloth: Installing prebuilt llama.cpp b10639-mix-f6f92fe (app-b10639-mix-f6f92fe-linux-x64-cpu.tar.gz) - skipping compilation.
Unsloth: Preparing converter script...
Unsloth: [1] Converting model into f16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['slm_audit_model_gguf/Qwen2.5-7B-Instruct.F16.gguf']
Unsloth: [2] Converting GGUF f16 into q4_k_m. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed 